# LLM server บน Colab (Gemma 4 E2B + Ollama + ngrok)

เครื่องผู้สอนเรียกมาที่นี่ด้วย REST + JSON (`POST /v1/chat`) เมื่อต้องการให้ GPU ของ Colab ตอบแทน CPU ถ้า Colab ช้า หลุด หรือล่ม gateway จะใช้ llama.cpp บนเครื่องผู้สอน (สำรอง) หรือกฎอ้างอิงเองโดยบอร์ดไม่ได้รับผลกระทบ

**วิธีใช้:** Runtime → Change runtime type → **T4 GPU** → Run all → คัดลอก "connection string" ที่เซลล์ 4 พิมพ์ ไปรันบนเครื่องผู้สอน

**ความลับ (ทำครั้งเดียว):** เปิด 🔑 Secrets ทางซ้าย เพิ่ม `NGROK_AUTHTOKEN` (จาก <https://dashboard.ngrok.com/get-started/your-authtoken>) และเปิด *Notebook access* อย่าเขียน token ลงในเซลล์

In [ ]:
#@title 1. ติดตั้ง Ollama และแพ็กเกจ
!nvidia-smi -L
!apt-get -qq update && apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install requests pyngrok

In [ ]:
#@title 2. ตั้งค่า, เปิด Ollama และดาวน์โหลดโมเดล
import os, secrets, subprocess, time, requests
MODEL = 'gemma4:e2b' #@param {type:'string'}
OLLAMA_URL = 'http://127.0.0.1:11434'
PORT = 5000
# สร้างใหม่ทุกครั้งที่รัน: ใครได้ URL ไปก็เรียกไม่ได้ถ้าไม่มี token
API_TOKEN = secrets.token_urlsafe(16)

os.environ.update({'OLLAMA_FLASH_ATTENTION': '1', 'OLLAMA_KV_CACHE_TYPE': 'q8_0', 'OLLAMA_NUM_PARALLEL': '1'})

def ollama_ready():
    try: return requests.get(OLLAMA_URL, timeout=2).status_code == 200
    except requests.RequestException: return False

if not ollama_ready():
    subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'),
                     stderr=subprocess.STDOUT, env=os.environ.copy())
    for _ in range(30):
        if ollama_ready(): break
        time.sleep(1)
    else: raise RuntimeError('Ollama did not start; see /content/ollama.log')
subprocess.run(['ollama', 'pull', MODEL], check=True)
print('Ollama ready with', MODEL)

In [ ]:
# Colab LLM server (stdlib only; the tests run this cell exactly as written)
#@title 3. REST server: POST /v1/chat
import hmac, json, threading, time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
import requests

MAX_BODY = 32768
MAX_HISTORY = 10
MAX_TEXT = 1000       # characters per message
MAX_SYSTEM = 2000
BUSY_SECONDS = 30     # wait this long for the GPU before answering 429

def parse_chat(body):
    """Validate the JSON body and return (messages, max_tokens, temperature)."""
    if not isinstance(body, dict) or not {'message'} <= set(body) <= {
            'system', 'history', 'message', 'max_tokens', 'temperature'}:
        raise ValueError('fields must be message plus optional system, history, max_tokens, temperature')
    system, message = body.get('system', ''), body['message']
    if not isinstance(system, str) or len(system) > MAX_SYSTEM:
        raise ValueError(f'system must be text up to {MAX_SYSTEM} characters')
    if not isinstance(message, str) or not 1 <= len(message.strip()) <= MAX_TEXT:
        raise ValueError(f'message must be 1-{MAX_TEXT} characters')
    history = body.get('history', [])
    if not isinstance(history, list) or len(history) > MAX_HISTORY:
        raise ValueError(f'history must be a list of at most {MAX_HISTORY} items')
    messages = [{'role': 'system', 'content': system}] if system.strip() else []
    for item in history:
        if (not isinstance(item, dict) or set(item) != {'role', 'content'}
                or item['role'] not in ('user', 'assistant')
                or not isinstance(item['content'], str) or not 1 <= len(item['content']) <= MAX_TEXT):
            raise ValueError('history items need role user/assistant and content text')
        messages.append(item)
    messages.append({'role': 'user', 'content': message})
    max_tokens, temperature = body.get('max_tokens', 200), body.get('temperature', 0.2)
    if type(max_tokens) is not int or not 1 <= max_tokens <= 512:
        raise ValueError('max_tokens must be an integer 1-512')
    if isinstance(temperature, bool) or not isinstance(temperature, (int, float)) or not 0 <= temperature <= 1:
        raise ValueError('temperature must be a number 0-1')
    return messages, max_tokens, float(temperature)

def make_server(port, api_token, model, ollama_url='http://127.0.0.1:11434'):
    gpu = threading.Lock()   # one request at a time on the T4, like OLLAMA_NUM_PARALLEL=1

    class Handler(BaseHTTPRequestHandler):
        protocol_version = 'HTTP/1.1'
        def log_message(self, *args): pass   # keep prompts out of the logs

        def reply(self, status, data):
            body = json.dumps(data, ensure_ascii=False).encode('utf-8')
            self.send_response(status)
            self.send_header('Content-Type', 'application/json; charset=utf-8')
            self.send_header('Content-Length', str(len(body)))
            self.send_header('Connection', 'close')
            self.end_headers()
            self.wfile.write(body)
            self.close_connection = True

        def do_GET(self):
            if self.path == '/health':
                self.reply(200, {'status': 'ok', 'model': model, 'busy': gpu.locked()})
            else:
                self.reply(404, {'error': 'not_found'})

        def do_POST(self):
            if self.path != '/v1/chat':
                return self.reply(404, {'error': 'not_found'})
            sent = self.headers.get('Authorization', '')
            if not hmac.compare_digest(sent.encode(), f'Bearer {api_token}'.encode()):
                return self.reply(401, {'error': 'unauthorized'})
            try:
                length = int(self.headers.get('Content-Length', '-1'))
            except ValueError:
                length = -1
            if not 0 <= length <= MAX_BODY:
                return self.reply(413 if length > MAX_BODY else 400, {'error': 'invalid_length'})
            try:
                messages, max_tokens, temperature = parse_chat(json.loads(self.rfile.read(length)))
            except (ValueError, UnicodeDecodeError) as exc:
                return self.reply(400, {'error': 'invalid_payload', 'detail': str(exc)})
            if not gpu.acquire(timeout=BUSY_SECONDS):
                return self.reply(429, {'error': 'busy'})
            started = time.monotonic()
            try:
                r = requests.post(f'{ollama_url}/api/chat', timeout=(5, 120), json={
                    'model': model, 'stream': False, 'think': False, 'keep_alive': -1,
                    'messages': messages,
                    'options': {'num_ctx': 4096, 'num_predict': max_tokens, 'temperature': temperature}})
                r.raise_for_status()
                text = ((r.json().get('message') or {}).get('content') or '').strip()
            except (requests.RequestException, ValueError) as exc:
                return self.reply(502, {'error': 'ollama_failed', 'detail': type(exc).__name__})
            finally:
                gpu.release()
            if not text:
                return self.reply(502, {'error': 'empty_reply'})
            self.reply(200, {'reply': text, 'model': model,
                             'latency_ms': int((time.monotonic() - started) * 1000)})

    server = ThreadingHTTPServer(('127.0.0.1', port), Handler)
    server.daemon_threads = True
    return server

In [ ]:
#@title 4. เปิด server และ ngrok tunnel
from google.colab import userdata
from pyngrok import ngrok

try: ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
except Exception as exc:
    raise RuntimeError('เพิ่ม NGROK_AUTHTOKEN ใน 🔑 Secrets และเปิด Notebook access ก่อน') from exc

for tunnel in ngrok.get_tunnels(): ngrok.disconnect(tunnel.public_url)   # รันซ้ำได้โดยไม่ซ้อน
server = make_server(PORT, API_TOKEN, MODEL, OLLAMA_URL)
threading.Thread(target=server.serve_forever, daemon=True).start()
PUBLIC_URL = ngrok.connect(PORT, 'http').public_url.replace('http://', 'https://')

# ทดสอบผ่านอินเทอร์เน็ตจริงเหมือนที่ gateway จะเรียก
HEADERS = {'ngrok-skip-browser-warning': '1', 'Authorization': f'Bearer {API_TOKEN}'}
print('health:', requests.get(PUBLIC_URL + '/health', headers=HEADERS, timeout=10).json())
t = time.monotonic()
test = requests.post(PUBLIC_URL + '/v1/chat', headers=HEADERS, timeout=120, json={
    'system': 'ตอบสั้นๆ เป็นภาษาไทย', 'history': [], 'message': 'ทักทายนักเรียนหนึ่งประโยค'}).json()
print(f'test reply ({time.monotonic() - t:.1f}s รวมโหลดโมเดลครั้งแรก):', test.get('reply', test))

print()
print('=== คัดลอกบรรทัดนี้ไปรันใน PowerShell บนเครื่องผู้สอน (ในโฟลเดอร์โปรเจกต์) ===')
print(f'.\\.venv\\Scripts\\python.exe -m gateway.remote "{PUBLIC_URL}#{API_TOKEN}"')
print('อย่าส่งบรรทัดนี้ให้นักเรียนหรือแคปหน้าจอแชร์: ใครมี token ก็ใช้ GPU นี้ได้')

In [ ]:
#@title 5. รันค้างไว้ตลอดคาบ (อย่าหยุดเซลล์นี้)
# Colab ตัดเซสชันที่ว่าง; เซลล์ที่ยังรันอยู่ช่วยกันไม่ให้หลุด
# ถ้า GPU ถูกถอนหรือเซสชันหมด gateway จะสลับไปใช้ llama.cpp บนเครื่องผู้สอนเอง
try:
    while True:
        h = requests.get('http://127.0.0.1:5000/health', timeout=5).json()
        print(time.strftime('%H:%M:%S'), 'ok', 'busy' if h['busy'] else 'idle', end='\r')
        time.sleep(30)
except KeyboardInterrupt:
    ngrok.kill(); server.shutdown(); print('\nหยุดแล้ว')